# Module 2 — Retrieval, grounding and evaluation


This is the complete teaching lab for Module 2. Run setup first, then follow the cells in order. Section headings are learning topics; [the current lesson map](../../docs/LESSON_MAP.md) maps them to the course videos. Collapsing a section does not skip execution.

**Practice versus assessment.** Worked examples teach the method; they are not completed capstone evidence. Use the [RiskLens capstone repository](https://github.com/SarasAI-Institute/Building-Custom-AI-Capstone) for exact evidence thresholds and all 24 Met / Not Yet criteria. Synthetic examples, tiny practice splits and unreviewed judge outputs do not establish mastery.

**Runtime.** Read [Colab setup and artifact handoffs](../../docs/COLAB.md). The CPU exercises run separately in `practice.ipynb` and `solutions.ipynb`. This lab uses model downloads and GPU execution. Modules 3–5 require your saved artifacts and explicit train/load choices. A notebook that stops for missing artifacts is not a completed run. See [what was tested](../../docs/VALIDATION.md).


## Shared setup before running

---
### 0 · Setup

`sentence-transformers` for embeddings, `faiss-cpu` for the vector index, `transformers` for
the generator LLM. *(FAISS = Facebook AI Similarity Search — the workhorse in-memory vector store.)*

In [ ]:
"""Run this cell first in a fresh Colab session, before importing model libraries."""
import importlib.metadata as _metadata
import platform as _platform
import subprocess as _subprocess
import sys as _sys
import tempfile as _tempfile
from pathlib import Path as _Path

COURSE_PACKAGES = [
    "transformers==5.22.0", "trl==1.17.0", "peft==0.21.2",
    "accelerate==1.15.0", "datasets==5.1.0", "sentence-transformers==6.2.0",
    "faiss-cpu==1.15.1", "num2words==0.5.14",
    "pillow==13.0.0", "scikit-learn==1.9.1", "matplotlib==3.11.2", "requests==2.34.2",
]
if _sys.version_info < (3, 11):
    raise RuntimeError("Use a current Colab Python 3.11+ runtime.")
try:
    _torch_version = _metadata.version("torch")
except _metadata.PackageNotFoundError as _error:
    raise RuntimeError("Use Colab's preinstalled matched PyTorch/torchvision runtime; see docs/COLAB.md.") from _error
_constraints = ["torch==" + _torch_version]
try:
    _constraints.append("torchvision==" + _metadata.version("torchvision"))
except _metadata.PackageNotFoundError:
    raise RuntimeError("The vision labs need a matching torchvision installation; use a fresh Colab runtime.")
if _platform.system() == "Linux":
    COURSE_PACKAGES.append("bitsandbytes==0.50.2")
_before = {}
for _name in ["transformers", "trl", "peft", "datasets", "sentence_transformers", "numpy"]:
    if _name in _sys.modules:
        _before[_name] = getattr(_sys.modules[_name], "__version__", None)
with _tempfile.TemporaryDirectory() as _temporary:
    _constraint_file = _Path(_temporary) / "preserve-colab-torch.txt"
    _constraint_file.write_text("\n".join(_constraints))
    _subprocess.run([_sys.executable, "-m", "pip", "install", "--quiet", "--upgrade-strategy", "only-if-needed",
                     "-c", str(_constraint_file), *COURSE_PACKAGES], check=True)
if _before:
    raise RuntimeError("Packages installed. Restart the Colab session once, then run from the setup cell again before importing models.")
print("Course stack installed; existing PyTorch/torchvision CUDA wheels preserved.")
print("If installation failed, stop here. Start a fresh GPU runtime and inspect the resolver message.")


In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
# T4s (pre-Ampere) have no native bfloat16 — detect and fall back to float16
DTYPE = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else (torch.float16 if torch.cuda.is_available() else torch.float32)
print("Using dtype:", DTYPE)

## Topic — What Is RAG? From Basic to Advanced Architectures

**Theory · 6–8 minutes**

Follows one question through retrieval, context assembly and generation. Compares a basic retrieve-then-generate pipeline with hybrid search plus reranking and an iterative, tool-directed retrieval loop. Explains when each adds value, and separates architecture choices from the particular vector database or model used.



### Visual companion

A question triggers retrieval; selected evidence enters a prompt; the generator produces an answer that must be checked against that evidence. Retrieval can use lexical, dense or hybrid methods. RAG does not require a particular vector database.

## Topic — Embeddings for Vector Search

**Theory · 6–8 minutes**

Explains how an embedding model represents queries and passages for semantic retrieval. Introduces similarity scores, cosine similarity and nearest-neighbor search, then distinguishes dense retrieval from keyword search and hybrid retrieval. Connects this to Module 1 while clarifying that token embeddings alone are not a document-search index.



### Visual companion

The notebook uses `BAAI/bge-small-en-v1.5`, a contextual retrieval encoder with 384-dimensional outputs. Its contextual CLS representation differs from averaging raw input embedding rows in Module 1. Follow this model’s query instruction convention: apply `BGE_PREFIX` to queries, not stored passages. Compatible dimensions alone do not make vectors from unrelated models comparable.

Both passage and query outputs are normalized. For nonzero unit vectors, inner product equals cosine similarity. `IndexFlatIP` performs exact exhaustive search under this score; it does not guarantee semantic relevance or answer support. A similarity score is not a calibrated answer probability. Check invalid or zero-norm outputs explicitly before indexing.

The video’s q=(1,0), K=(0.8,0.6), T=(0.6,0.8), B=(−0.6,0.8) are constructed 2D teaching values, not BGE measurements or projections. Their cosine scores are 0.8, 0.6, −0.6; squared unit distances are 0.4, 0.8, 3.2. Dense and lexical search can complement one another for the same query, but that improvement must be evaluated.


## Topic — Chunking Strategies & Retrieval Trade-Offs

**Theory · 6–8 minutes**

Compares fixed-size windows, overlapping chunks and structure-aware splitting by headings or paragraphs. Introduces semantic and parent-child chunking as extensions. Uses one review/specification example to explain the trade-off between precise retrieval, sufficient context, metadata preservation and the generator context budget.



### Visual companion

A chunk boundary changes which claims and qualifiers stay together. The workshop defaults remain `max_words=120` and `overlap=30`, measured in whitespace-delimited words. There is no universal 100–300-word optimum, and word counts are not tokenizer counts. Check the encoder and generator with their own tokenizers and limits.

Overlap can recover a short phrase across a boundary, but it cannot guarantee that a whole sentence or dependency survives inside one chunk. In the video’s exact eighteen-word service-note excerpt, ten-word windows with overlap four cover 1–10, 7–16 and 13–18 (one-based inclusive positions). They contain 26 word instances for 18 unique positions. The phrase 9–12 is together in the middle chunk; the date condition 1–4 and part number 17–18 still fit no single ten-word chunk.

Structure-aware boundaries, semantic proposals and parent-child expansion are alternatives to evaluate, not automatic improvements. Keep source identity and offsets; deduplicate shared parents before context assembly. The budget diagram is illustrative and includes metadata inside the evidence blocks.

The the corresponding topic splitter validates `0 <= overlap < max_words`, skips empty text and stops when a window reaches the end. It adds `chunk_id`, `start_word` and `end_word` while retaining `source` and `text`. Notebook offsets are zero-based and end-exclusive, unlike the video’s one-based labels. This prevents redundant contained tails in longer inputs. Parent-child and semantic splitters remain conceptual extensions; they are not silently introduced as dependencies.


## Topic — From Reviews & Specs to a Working Retriever

**Tutorial: Colab · 6–8 minutes**

Prepares a small corpus with source identifiers and freezes gold questions before retrieval experiments. Shows one chunking choice, builds the embedding index and inspects the returned passages.

Run Shared setup first. Freeze source text and the six practice gold rows before building the index. Save the snapshot file outside the Colab runtime. These questions exercise synthetic specs and policy; they do not validate every review task. Inspect complete returned passages and source IDs. The index uses exact normalized inner-product search; scores are not probabilities.


### The knowledge base

Stream a small set of real appliance reviews and combine them with four explicitly synthetic course spec/policy documents. These documents provide the exercise’s reference facts. The reviews are not automatically about KT-2000, TS-400 or BL-900; do not join them to those invented models by topic similarity. In a real system, preserve verified product identifiers and provenance before combining evidence.


In [ ]:
from datasets import load_dataset
from huggingface_hub import HfApi
import itertools
REVIEW_DATASET="McAuley-Lab/Amazon-Reviews-2023"
REVIEW_REVISION=HfApi().dataset_info(REVIEW_DATASET).sha
REVIEW_URL=f"https://huggingface.co/datasets/{REVIEW_DATASET}/resolve/{REVIEW_REVISION}/raw/review_categories/Appliances.jsonl"
reviews_raw=load_dataset("json",data_files=REVIEW_URL,split="train",streaming=True)
reviews=[]
for source_row,r in enumerate(itertools.islice(reviews_raw,20000)):
    text=r.get("text","")
    if isinstance(text,str) and 200<len(text)<1200:
        reviews.append({"source":f"review_{len(reviews):03d}","text":text.strip(),"kind":"review",
            "provenance":{"dataset":REVIEW_DATASET,"revision":REVIEW_REVISION,"url":REVIEW_URL,
            "source_row":source_row,"parent_asin":r.get("parent_asin"),"timestamp":r.get("timestamp")}})
    if len(reviews)==120: break
if len(reviews)<120: raise RuntimeError("Fewer than 120 eligible reviews in the bounded scan; inspect the data source.")
print("Collected",len(reviews),"real reviews. Do not map them to fictional catalogue products.")


And a handful of internal spec documents — deliberately containing facts the reviews
*don't* have (warranty terms, materials, part numbers), so we can later ask questions that
require both sources:

In [ ]:
specs = [
    {"source": "spec_kettle",  "text": "Model KT-2000 electric kettle. Capacity 1.7L. Body: borosilicate "
        "glass with a polypropylene lid assembly. Lid hinge part number PP-114. Boil-dry protection "
        "included. Warranty: 24 months covering material and manufacturing defects."},
    {"source": "spec_toaster", "text": "Model TS-400 two-slot toaster. Stainless steel housing. Crumb tray "
        "part number CT-security-09. Known service note: units manufactured before 2023-06 may exhibit "
        "lever sticking; resolved by revision C of the spring assembly, part SP-77C."},
    {"source": "spec_blender", "text": "Model BL-900 blender. 900W motor, tritan jar 1.5L. Blade assembly "
        "part number BA-45. The jar is NOT dishwasher safe above 60°C. Warranty: 12 months, motor only."},
    {"source": "policy_returns", "text": "Return policy for all products (models KT-2000, TS-400, BL-900): "
        "defective units can be returned within 30 days for a full refund. Warranty claims after 30 days "
        "are handled as repair or replacement. Water damage and misuse are excluded from warranty coverage."},
]

corpus = reviews + specs
print(f"Total corpus: {len(corpus)} documents")

### Freeze the gold questions before indexing

Read the corpus first. The six rows below are a small practice set. Keep them fixed across retrieval comparisons; use development questions to choose changes and preserve a separate final evaluation set for a defensible project.

In [ ]:
gold = [
    {"q": "What is the warranty period for the KT-2000 kettle?",
     "sources": ["spec_kettle"],  "ref": "24 months"},
    {"q": "What part number is the kettle lid hinge?",
     "sources": ["spec_kettle"],  "ref": "PP-114"},
    {"q": "Is the blender jar dishwasher safe?",
     "sources": ["spec_blender"], "ref": "Not dishwasher safe above 60°C; safety at or below 60°C is not established by this statement alone."},
    {"q": "Which spring assembly revision fixed the toaster lever sticking?",
     "sources": ["spec_toaster"], "ref": "revision C, part SP-77C"},
    {"q": "How long do customers have to return a defective unit for a refund?",
     "sources": ["policy_returns"], "ref": "30 days"},
    {"q": "Does the warranty cover water damage?",
     "sources": ["policy_returns"], "ref": "excluded from coverage"},
]
print(f"Gold set: {len(gold)} questions — FROZEN 🔒 (in the capstone: ≥20, written before building)")

In [ ]:
import hashlib, json
from pathlib import Path

# Freeze content and labels before experiments. Save/download this file from Colab.
snapshot = {"corpus": corpus, "gold": gold}
snapshot_sha256 = hashlib.sha256(json.dumps(snapshot, sort_keys=True, ensure_ascii=False).encode()).hexdigest()
freeze_path = Path("rag_practice_snapshot_v1.json")
if freeze_path.exists():
    previous = json.loads(freeze_path.read_text())
    if previous["sha256"] != snapshot_sha256:
        raise ValueError("Corpus or gold changed. Restore the saved snapshot or deliberately create a new experiment version.")
else:
    freeze_path.write_text(json.dumps({"version": 1, "sha256": snapshot_sha256, **snapshot}, indent=2, ensure_ascii=False))
assert len({d["source"] for d in corpus}) == len(corpus)
assert all(set(g["sources"]) <= {d["source"] for d in corpus} for g in gold)
print("Frozen practice snapshot:", snapshot_sha256, "documents:", len(corpus), "questions:", len(gold))


In [ ]:
def chunk(text, source, max_words=120, overlap=30):
    # Word offsets are zero-based and end-exclusive; these are not tokenizer offsets.
    if type(max_words) is not int or type(overlap) is not int:
        raise TypeError("max_words and overlap must be integers")
    if max_words <= 0 or not 0 <= overlap < max_words:
        raise ValueError("Require max_words > 0 and 0 <= overlap < max_words")
    words = text.split()
    if not words:
        return []
    out, start = [], 0
    while start < len(words):
        end = min(start + max_words, len(words))
        piece = text if start == 0 and end == len(words) else " ".join(words[start:end])
        out.append({"source": source, "text": piece,
                    "chunk_id": f"{source}:{start}:{end}",
                    "start_word": start, "end_word": end})
        if end == len(words):
            break  # Do not add a tail already contained in this final window.
        start += max_words - overlap
    return out

chunks = [{**c,"kind":doc.get("kind","doc")} for doc in corpus for c in chunk(doc["text"], doc["source"])]
print(f"{len(corpus)} documents → {len(chunks)} chunks")
for c in chunks[:3]:
    print(c["chunk_id"], c["start_word"], c["end_word"], c["text"][:90])


In [ ]:
from sentence_transformers import SentenceTransformer

from huggingface_hub import model_info
EMBEDDING_REVISION=model_info("BAAI/bge-small-en-v1.5").sha
embedder=SentenceTransformer("BAAI/bge-small-en-v1.5",revision=EMBEDDING_REVISION,device="cpu")

vectors = embedder.encode(
    [c["text"] for c in chunks],
    normalize_embeddings=True,     # unit length → dot product == cosine similarity
    show_progress_bar=True,
)
print("Embedding matrix:", vectors.shape)   # (num_chunks, 384)

Every chunk is now a point in 384-dimensional space. Searching = *"embed the question,
find the nearest points."* FAISS does that lookup efficiently:

In [ ]:
import faiss
import numpy as np

vectors = np.ascontiguousarray(vectors, dtype=np.float32)
if vectors.ndim != 2 or len(vectors) != len(chunks) or not len(chunks):
    raise ValueError("Require one nonempty embedding row per chunk.")
if not np.isfinite(vectors).all() or not np.allclose(np.linalg.norm(vectors, axis=1), 1, atol=1e-4):
    raise ValueError("Passage embeddings must be finite and unit-normalized.")
index = faiss.IndexFlatIP(vectors.shape[1])  # Exact inner-product search; unit vectors give cosine.
index.add(vectors)
BGE_PREFIX = "Represent this sentence for searching relevant passages: "

def retrieve(question, k=4):
    if not isinstance(question, str) or not question.strip():
        raise ValueError("Supply a nonempty question.")
    if type(k) is not int or k < 1:
        raise ValueError("k must be a positive integer.")
    if index.ntotal == 0:
        return []
    q = np.ascontiguousarray(embedder.encode([BGE_PREFIX + question], normalize_embeddings=True), dtype=np.float32)
    if q.shape != (1, index.d) or not np.isfinite(q).all() or not np.allclose(np.linalg.norm(q, axis=1), 1, atol=1e-4):
        raise ValueError("Query embedding must be finite, unit-normalized and dimension-compatible.")
    scores, idxs = index.search(q, min(k, index.ntotal))
    return [{**chunks[int(i)], "score": float(s)} for s, i in zip(scores[0], idxs[0]) if 0 <= i < len(chunks)]

for hit in retrieve("kettle lid broke off", k=3):
    print(f"[{hit['score']:.3f}] ({hit['source']}) {hit['chunk_id']}\n{hit['text']}\n")


Inspect the returned passages and check whether they support the question. Dense retrieval may find related wording, but relevance and evidence coverage must be measured. **The generator has not been called yet.** The embedding encoder is itself a learned language model. Retrieval cost and latency depend on the encoder, corpus, index and hardware; evaluate them separately from generation.



## Topic — Generate Cited Answers & Refuse Unsupported Questions

**Tutorial: Colab · 6–8 minutes**

Connects retrieved passages to the generator, demonstrates a supported answer and an unanswerable question, and checks whether citations support the claims. Compares the same question with and without retrieved context.

Requires the corresponding topic corpus snapshot, chunks, embedder and retrieve. Prepare the generator before filming. The generator helper checks the actual chat-template length plus output reserve against an explicit 2,048-token teaching cap, bounded by the model configuration. Save raw answers and exact contexts; manual support judgments remain pending until inspected.


### 4 · The G in RAG: grounded generation

Assemble the prompt from selected passages, source IDs, the question and explicit instructions:

1. **Answer from the supplied context.** This is an instruction, not a guarantee against unsupported claims.
2. **Cite sources.** Verify that each cited passage actually supports its attached claim.
3. **State when the context is insufficient.** Preserve a limited answer instead of inventing missing evidence.

Generator: `Qwen2.5-1.5B-Instruct`. The workshop uses this small generator for a Colab demonstration. Model size alone does not establish adequate quality, speed or hardware fit. Inspect retrieval coverage and generation faithfulness separately; either stage can limit the system. Run the current notebook in the intended Colab environment before running.


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

GEN_ID = "Qwen/Qwen2.5-1.5B-Instruct"
GEN_REVISION=model_info(GEN_ID).sha
gen_tok=AutoTokenizer.from_pretrained(GEN_ID,revision=GEN_REVISION)
gen_lm = AutoModelForCausalLM.from_pretrained(
    GEN_ID, revision=GEN_REVISION, dtype=DTYPE, device_map={"":0 if torch.cuda.is_available() else "cpu"}
)
print(f"Generator loaded: {gen_lm.num_parameters()/1e9:.2f}B params")

In [ ]:
GEN_CONTEXT_BUDGET = min(2048, int(gen_lm.config.max_position_embeddings))  # Explicit teaching cap.

def generate(messages, max_new_tokens=300):
    """Decode only new tokens; reject prompt/output budgets that exceed the teaching cap."""
    if type(max_new_tokens) is not int or max_new_tokens < 1:
        raise ValueError("max_new_tokens must be a positive integer.")
    inputs = gen_tok.apply_chat_template(
        messages, add_generation_prompt=True, return_dict=True, return_tensors="pt"
    ).to(gen_lm.device)
    prompt_tokens = inputs["input_ids"].shape[1]
    if prompt_tokens + max_new_tokens > GEN_CONTEXT_BUDGET:
        raise ValueError(f"Prompt {prompt_tokens} + reserved output {max_new_tokens} exceeds {GEN_CONTEXT_BUDGET}. Reduce or select evidence explicitly.")
    gen_lm.eval()
    with torch.inference_mode():
        out = gen_lm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                              pad_token_id=gen_tok.eos_token_id)
    return gen_tok.decode(out[0][prompt_tokens:], skip_special_tokens=True).strip()


In [ ]:
def rag_answer(question, k=4):
    hits = retrieve(question, k)
    context = "\n\n".join(f"[{h['source']}]\n{h['text']}" for h in hits)

    system = (
        "You answer questions about our products using ONLY the provided context. "
        "Cite the source id in brackets after each claim, e.g. [spec_kettle]. "
        "If the context does not contain the answer, say exactly: "
        "'I don't have enough information to answer that.'"
    )
    user = f"Context:\n{context}\n\nQuestion: {question}"
    return generate([{"role": "system", "content": system},
                     {"role": "user", "content": user}]), hits

### Inspect support and insufficient information

Try a specification question, a summary of only the retrieved review excerpts, and a personal-unit question the context cannot establish. The instruction requests refusal; the actual model may fail. Inspect every claim against the exact returned text. A citation ID that exists is only a traceability check, not evidence that the claim follows. Top-k review retrieval cannot establish the most frequent problems across all customers.


In [ ]:
import re

# An ID-presence check catches missing/unknown labels; it does NOT prove claim support.
def citation_ids(answer):
    return re.findall(r"\[([A-Za-z0-9_.:-]+)\]", answer)

demo_records = []
for question in [
    "What is the warranty on the KT-2000 kettle?",
    "What problems are described in the supplied review excerpts? Do not estimate how common they are.",
    "What is the exact manufacture date of my own KT-2000 unit?",
]:
    answer, hits = rag_answer(question)
    available = {h["source"] for h in hits}
    cited = citation_ids(answer)
    record = {"question": question, "answer": answer, "hits": hits,
              "cited_ids": cited, "unknown_cited_ids": sorted(set(cited) - available),
              "claim_support_review": "PENDING HUMAN REVIEW"}
    demo_records.append(record)
    print(f"Q: {question}\nA: {answer}\nUnknown citation IDs: {record['unknown_cited_ids']}")
    for hit in hits:
        print(f"[{hit['source']}] {hit['chunk_id']}\n{hit['text']}\n")
    print("Check each claim against these exact passages; verify the expected refusal, do not assume it.")
    print("-" * 70)
Path("rag_generation_demo.json").write_text(json.dumps({"snapshot_sha256": snapshot_sha256,
    "generator_id": GEN_ID, "generator_revision": GEN_REVISION,
    "context_budget": GEN_CONTEXT_BUDGET, "records": demo_records}, indent=2, ensure_ascii=False))


### Compare the same question with and without retrieved context

Keep the generator and question fixed. This demonstration changes both supplied context and the grounding instructions; it illustrates two pipelines rather than isolating retrieval as the only causal variable. Preserve both complete outputs, including failures, and inspect them against the source.


In [ ]:
q = "Is the BL-900 blender jar dishwasher safe?"
bare = generate([{"role": "user", "content": q}])
grounded, comparison_hits = rag_answer(q)
print("WITHOUT retrieval:\n", bare)
print("\nWITH retrieval:\n", grounded)
for hit in comparison_hits:
    print(f"[{hit['source']}] {hit['text']}")
Path("rag_with_without_comparison.json").write_text(json.dumps({
    "snapshot_sha256": snapshot_sha256, "generator_id": GEN_ID, "question": q,
    "bare_answer": bare, "grounded_answer": grounded, "hits": comparison_hits,
    "interpretation": "Pipeline demonstration; context and system instructions differ, so this is not an isolated retrieval ablation."
}, indent=2, ensure_ascii=False))


### What the comparison establishes

A no-context model may answer from prior knowledge, abstain or hallucinate. Retrieval supplies inspectable evidence; it does not guarantee the answer uses that evidence correctly. Check each material claim against its cited passage, and test unsupported questions explicitly.

## Topic — When Graphs Help: GraphRAG & KAG

**Theory · 8 minutes**

Trace a question whose service-part and return-policy evidence spans linked sources. Follow one projected source-neighbor step, inspect the actual policy qualifiers, and distinguish entity overlap from Microsoft GraphRAG and OpenSPG KAG. The animation's links and seed are curated teaching examples, not measured model output. The following live cells retain raw extraction traces and retrieval provenance for inspection.


In [ ]:
import json, re

extraction_trace = []
def extract_entities(text, source=None):
    """Inspect a small model-generated entity map; this is one possible extraction method."""
    prompt = (
        "List the product models, part numbers, and defect types mentioned in this text. "
        'Respond ONLY with JSON: {"entities": ["...", "..."]}\n\nText: ' + text
    )
    raw = generate([{"role": "user", "content": prompt}], max_new_tokens=100)
    record = {"source": source, "raw": raw, "status": "unreviewed", "entities": []}
    try:
        match = re.search(r"\{.*\}", raw, re.DOTALL)
        if match is None:
            raise ValueError("No JSON object found; output may be truncated.")
        entities = json.loads(match.group())["entities"]
        if not isinstance(entities, list) or any(not isinstance(e, str) or not e.strip() for e in entities):
            raise ValueError("Expected a list of nonempty entity strings.")
        record["entities"] = sorted({e.strip().lower() for e in entities})
        record["status"] = "parsed; source mapping still needs human review"
    except (ValueError, KeyError, TypeError) as exc:
        record["status"] = "parse failure"
        record["error"] = str(exc)
        print(f"Extraction failed for {source}: {exc}")
    extraction_trace.append(record)
    return record["entities"]

# Lowercasing is a minimal normalization, not entity resolution.
graph = {}
for doc in specs:
    for ent in extract_entities(doc["text"], source=doc["source"]):
        graph.setdefault(ent, set()).add(doc["source"])
Path("graph_extraction_trace.json").write_text(json.dumps({
    "snapshot_sha256": snapshot_sha256, "records": extraction_trace,
    "graph": {e: sorted(s) for e, s in graph.items()}
}, indent=2, ensure_ascii=False))
for ent, sources in list(graph.items())[:8]:
    print(f"{ent:30} → {sorted(sources)}")
print("Review raw outputs and source mentions. Missing or malformed extraction is not evidence that an entity is absent.")


Now **graph-augmented retrieval**: retrieve seed chunks, then add source documents sharing an extracted entity with an original seed source. This is one step in the projected source-neighbor graph, or two edges in the source–entity incidence picture. Newly added sources do not seed another expansion. Keep the original dense score and record the retrieval method separately: valid cosine scores can be negative. Graph additions have no computed similarity score.


In [ ]:
def graph_retrieve(question, k=3):
    hits = [{**h, "retrieval_method": "dense", "via_entities": []} for h in retrieve(question, k)]
    seed_sources = {h["source"] for h in hits}  # Keep this fixed for exactly one expansion.
    via = {}
    for entity, sources in graph.items():
        if seed_sources & set(sources):
            for source in set(sources) - seed_sources:
                via.setdefault(source, set()).add(entity)
    seen = {h["chunk_id"] for h in hits}
    for chunk in chunks:
        if chunk["source"] in via and chunk["chunk_id"] not in seen:
            hits.append({**chunk, "score": None, "retrieval_method": "graph",
                         "via_entities": sorted(via[chunk["source"]])})
            seen.add(chunk["chunk_id"])
    return hits

q = ("Which part fixes the toaster lever sticking issue, and does the policy allow "
     "a full refund for a customer who bought one 90 days ago?")
for h in graph_retrieve(q):
    tag = h["retrieval_method"]
    score = "not computed" if h["score"] is None else f"{h['score']:.3f}"
    print(f"[{tag}] {h['source']} score={score} via={h['via_entities']}\n{h['text']}\n")


### Scope of this demonstration

This small entity/source overlap expansion is not a full implementation of Microsoft GraphRAG or OpenSPG KAG. Microsoft GraphRAG includes entity/relationship extraction, community reports and different local/global query machinery. KAG adds logical-form-guided retrieval and reasoning operators. These are design choices to evaluate, not a guaranteed quality ladder.

The diagram uses a curated TS-400 map and a constructed seed to make the mechanism reproducible; actual model extraction and retrieval may differ. Inspect `graph_extraction_trace.json`, check source mentions and retain failures. Lowercasing does not solve aliases or entity resolution. High-degree entities can add unrelated evidence. Compare supported answers, missing evidence, context length and cost on fixed questions; do not infer quality from source count alone.

The helper deduplicates chunks and tags provenance explicitly. It can append many chunks and exceed a generator budget, so select and serialize context explicitly before generation. A path supplies candidate evidence; it does not prove a claim or establish an individual customer's eligibility.

References: [Microsoft indexing](https://microsoft.github.io/graphrag/index/overview/), [query modes](https://microsoft.github.io/graphrag/query/overview/), [OpenSPG KAG](https://github.com/OpenSPG/KAG).


## Topic — RAG Evaluation: Retrieval, Grounding & Answer Quality

**Theory · 8 minutes**

Trace four separate objects: required source IDs, retrieved chunks and their exact text, generated answer claims, and the question. Source coverage, faithfulness, answer relevance and citation support compare different links. The animation uses constructed rankings and responses, not measured model scores. Review failures at each link before changing a pipeline component.


### Define each metric before comparing systems

For question $q$, let $G_q$ be the nonempty set of required source IDs and $S_q(k)$ the distinct source IDs in the first $k$ retrieved chunks. Fractional source recall is $|G_q \cap S_q(k)|/|G_q|$. All-source success is the indicator $\mathbf{1}[G_q\subseteq S_q(k)]$. The original workshop's `recall_at_k` averaged the latter; these definitions coincide on its single-source gold questions. Count a source once even when several chunks come from it. Source ID coverage alone does not ensure that the retrieved passage contains the answer.

The animation uses required sources {T,P} and chunk order T1,K1,T2,P1. At k=2, fractional recall is 1/2 and all-source success is 0. With a second fully covered single-source question, macro recall is 0.75, all-source success is 0.50 and pooled source recall is 2/3. State which aggregation you use. These are constructed arithmetic examples.

**Faithfulness** checks support in the supplied context; **answer relevance** checks whether the response addresses the question. An answer about the kettle's 1.7 L capacity can be supported but irrelevant to a warranty question. A 12-month kettle warranty can address that question but contradict the synthetic source's 24 months. A citation must point to the passage supporting its attached claim: an existing ID is insufficient.

The lesson's three manual claim judgments are supported, supported, unsupported. Claim-level support is 2/3; an all-claims-supported binary verdict is 0. These aggregations differ. Ragas faithfulness uses a claim-support proportion; its answer-relevancy metric uses generated questions and embedding similarities. The notebook's lightweight YES/NO prompts are teaching probes, not equivalent Ragas implementations or calibrated measures.

Questions with no answer in the corpus need a separate refusal check. Empty gold sets make recall undefined and subset tests vacuously true. Distinguish corpus absence from retrieval failure, and count unnecessary refusals on answerable questions. Preserve raw judge output; exact YES/NO parsing prevents a NOT YES substring bug but cannot make the judge correct. Review calibration cases manually. The existing six gold questions are development practice, not a held-out final test.

References: [Ragas faithfulness](https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/faithfulness/), [Ragas answer relevancy](https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/answer_relevance/).


## Topic — Evaluate RAG & Choose a Retrieval Upgrade

**Tutorial: Colab · 6–8 minutes**

Read the source-recall and all-source-success table, inspect faithfulness and relevance separately, then compare dense retrieval at k=2 and k=4 on the same development cases. Prepare the model run before filming; retain complete raw outputs and failures. Inspect the personal-unit question for refusal behavior. Complete the human review and adopt-or-keep conclusion only after examining actual evidence. No guaranteed improvement is assumed.


In [ ]:
def source_scores(required_sources, hits):
    required = set(required_sources)
    found = {h["source"] for h in hits}
    if not required:
        return {"source_recall": None, "all_source_success": None,
                "missing_sources": [], "answerability_review": "separate refusal case"}
    return {"source_recall": len(required & found) / len(required),
            "all_source_success": int(required <= found),
            "missing_sources": sorted(required - found)}

def retrieval_table(gold_set, k=4, retriever=None):
    if type(k) is not int or k < 1:
        raise ValueError("k must be a positive integer")
    retriever = retrieve if retriever is None else retriever
    rows = [{"question": item["q"], "required_sources": item["sources"],
             "hits": (hits := retriever(item["q"], k)),
             **source_scores(item["sources"], hits)} for item in gold_set]
    eligible = [x for x in rows if x["source_recall"] is not None]
    return {"k": k, "eligible_questions": len(eligible),
            "macro_source_recall": sum(x["source_recall"] for x in eligible) / len(eligible) if eligible else None,
            "all_source_success": sum(x["all_source_success"] for x in eligible) / len(eligible) if eligible else None,
            "rows": rows}

def recall_at_k(gold_set, k=4):
    """Legacy workshop behavior: mean ALL-source success, now explicitly named in reports."""
    return retrieval_table(gold_set, k)["all_source_success"]

retrieval_results = [retrieval_table(gold, k) for k in [1, 2, 4, 8]]
for result in retrieval_results:
    print({key: value for key, value in result.items() if key != "rows"})
Path("rag_retrieval_table.json").write_text(json.dumps({"snapshot_sha256": snapshot_sha256,
    "results": retrieval_results}, indent=2, ensure_ascii=False))


A larger k can recover more required sources but also adds context. The table reports **macro fractional source recall** and **mean all-source success** separately. Both exclude questions with empty required-source sets; report their refusal behavior separately. Keep all hit text, not just IDs. The original six single-source questions make the two measures coincide, so the theory's constructed two-source example matters.


In [ ]:
def parse_judge(raw):
    verdict = raw.strip().upper()
    return {"raw": raw, "verdict": verdict if verdict in {"YES", "NO"} else "INVALID",
            "score": {"YES": 1, "NO": 0}.get(verdict)}

def judge_faithfulness(question, answer, context):
    """Uncalibrated all-claims-supported probe, not a Ragas claim fraction."""
    prompt = ("You are a strict evaluator. Context:\n" + context +
        f"\n\nQuestion: {question}\nAnswer: {answer}\n\n"
        "Is every factual claim in the Answer directly supported by the Context? "
        "Reply with exactly one word: YES or NO.")
    raw = generate([{"role": "user", "content": prompt}], max_new_tokens=8)
    return {**parse_judge(raw), "prompt": prompt}


In [ ]:
def judge_relevance(question, answer):
    """Uncalibrated binary topic/answer probe, not Ragas answer relevancy."""
    prompt = (f"Question: {question}\nAnswer: {answer}\n\n"
        "Does the Answer directly address the Question, regardless of factual correctness? "
        "Reply with exactly one word: YES or NO.")
    raw = generate([{"role": "user", "content": prompt}], max_new_tokens=8)
    return {**parse_judge(raw), "prompt": prompt}


Faithfulness and answer relevance check different relationships; this does not assert statistical independence. The following binary probes are not Ragas implementations. Parse exact YES/NO and retain INVALID, ERROR and raw verdicts. A well-formatted verdict can still be wrong. Human review of claims, citations, refusal and correctness is required; generator/judge shared weights can share errors.


In [ ]:
import time

# These are development cases. Freeze this comparison plan before looking at its outputs.
comparison_cases = [{**item, "answerable": True} for item in gold] + [{
    "q": "What is the exact manufacture date of my own KT-2000 unit?",
    "sources": [], "ref": "The corpus cannot establish this individual unit's date.",
    "answerable": False}]
comparison_plan = {"snapshot_sha256": snapshot_sha256, "cases": comparison_cases,
    "baseline_k": 2, "candidate_k": 4, "change": "dense retrieval depth only",
    "generator_id": GEN_ID, "generator_revision": GEN_REVISION,
    "context_budget": GEN_CONTEXT_BUDGET, "decoding": "greedy; max_new_tokens=300",
    "decision_rule": "Adopt only if reviewed answer quality improves and extra context/time are justified; otherwise keep baseline.",
    "evaluation_role": "development practice, not held-out final evaluation"}
plan_path = Path("rag_comparison_plan.json")
if plan_path.exists():
    if json.loads(plan_path.read_text()) != comparison_plan:
        raise ValueError("Comparison plan changed. Save a new named experiment; do not overwrite an old comparison.")
else:
    plan_path.write_text(json.dumps(comparison_plan, indent=2, ensure_ascii=False))

def run_comparison_case(item, k):
    row = {"question": item["q"], "answerable": item["answerable"], "reference": item["ref"],
           "required_sources": item["sources"], "k": k, "manual_review": {
               "correctness": None, "claim_support": None, "answer_relevance": None,
               "citation_support": None, "refusal_appropriate": None, "notes": None}}
    started = time.perf_counter()
    try:
        answer, hits = rag_answer(item["q"], k=k)
        row.update({"status": "generated", "answer": answer, "hits": hits,
                    "retrieval_and_generation_seconds": time.perf_counter() - started,
                    **source_scores(item["sources"], hits)})
        context = "\n\n".join(f"[{h['source']}]\n{h['text']}" for h in hits)
        row["context"] = context
        row["context_words"] = len(context.split())  # Not model tokens or total prompt length.
        row["cited_ids"] = citation_ids(answer)
        row["unknown_cited_ids"] = sorted(set(row["cited_ids"]) - {h["source"] for h in hits})
        for name, call in [("faithfulness_probe", lambda: judge_faithfulness(item["q"], answer, context)),
                           ("relevance_probe", lambda: judge_relevance(item["q"], answer))]:
            try:
                row[name] = call()
            except Exception as exc:
                row[name] = {"verdict": "ERROR", "score": None, "error": repr(exc)}
    except Exception as exc:
        row.update({"status": "generation_error", "error": repr(exc),
                    "retrieval_and_generation_seconds": time.perf_counter() - started})
    return row

comparison_path = Path("rag_k2_k4_comparison.json")
if comparison_path.exists():
    saved = json.loads(comparison_path.read_text())
    if saved["plan"] != comparison_plan:
        raise ValueError("Saved outputs belong to a different plan.")
    comparison_rows = saved["rows"]
    print("Loaded saved outputs; no model calls repeated. Preserve this run before starting another.")
else:
    comparison_rows = []
    for item in comparison_cases:
        for k in [2, 4]:
            row = run_comparison_case(item, k)
            comparison_rows.append(row)
            comparison_path.write_text(json.dumps({"plan": comparison_plan, "rows": comparison_rows,
                "status": "incomplete" if len(comparison_rows) < 2 * len(comparison_cases) else "complete; manual review pending"},
                indent=2, ensure_ascii=False))
    saved = json.loads(comparison_path.read_text())
if len(comparison_rows) != 2 * len(comparison_cases):
    raise RuntimeError("Saved run is incomplete. Inspect its errors and preserve it before explicitly starting a new run.")
for row in comparison_rows:
    print(row["question"], "k=", row["k"], row["status"],
          "source recall=", row.get("source_recall"), "all found=", row.get("all_source_success"))
    print(row.get("answer", row.get("error")))
    print("Judges:", row.get("faithfulness_probe"), row.get("relevance_probe"))
    print("Unknown citation IDs:", row.get("unknown_cited_ids"))
    print("Review exact passages:", row.get("context", "No generated context saved"))
    print("-" * 70)
print("Raw results saved. No reference-word substring is used as a correctness score.")


### Review the change and state a decision

The comparison changes retrieval depth only: k=2 baseline, k=4 candidate. Corpus, questions, encoder/index, generator, grounding prompt and decoding settings remain fixed. Source coverage may improve while answer quality stays flat or gets worse. Context word count is not model-token count. A single end-to-end timing per case is diagnostic, not a latency benchmark; it excludes judge calls and is sensitive to order, hardware and warmup.

Inspect the unsupported personal-unit question even if a judge calls its answer relevant. A specific invented date is a refusal failure; withholding the date is appropriate. If no actual failure occurs, say so. The theory's constructed failure is a separate teaching example, not a measured result. For answerable cases, review unnecessary refusals too.

Open `rag_comparison_human_review.json` and record each case's correctness, claim support, answer relevance, citation support, refusal appropriateness and evidence notes. Use explicit not-applicable entries when warranted. State ADOPT k=4 only if the actual reviewed benefit justifies added context/time; otherwise KEEP k=2. Preserve errors, invalid judges and complete raw outputs. Do not pick a winner from source recall alone.

The six original gold questions and added unsupported case are development practice. Project-scale evaluation needs a separate authored and frozen test set. Additional chunking, hybrid or graph sweeps remain extensions; measure them rather than claiming they ran. New failures can enter a future development set without contaminating an existing final test.


In [ ]:
# Inspect both outputs and exact passages, then fill the separate human review file.
review_path = Path("rag_comparison_human_review.json")
if not review_path.exists():
    review_path.write_text(json.dumps({"snapshot_sha256": snapshot_sha256,
        "reviews": [{"question": x["question"], "k": x["k"], **x["manual_review"]} for x in comparison_rows],
        "decision": "PENDING", "reason": "Review actual answers, refusals, citations, failures and extra context/time before choosing."},
        indent=2, ensure_ascii=False))
review = json.loads(review_path.read_text())
if review["decision"] not in {"PENDING", "ADOPT k=4", "KEEP k=2"}:
    raise ValueError("Use PENDING, ADOPT k=4 or KEEP k=2 and explain the evidence.")
if review["decision"] != "PENDING":
    expected = {(x["question"], x["k"]) for x in comparison_rows}
    actual = {(x["question"], x["k"]) for x in review["reviews"]}
    if expected != actual or len(review["reviews"]) != len(expected):
        raise ValueError("Review every baseline/candidate case exactly once.")
    if any(any(x.get(key) is None for key in ["correctness", "claim_support", "answer_relevance", "citation_support", "refusal_appropriate", "notes"]) for x in review["reviews"]):
        raise ValueError("Complete each review; use a justified not-applicable entry where appropriate.")
    if not str(review.get("reason", "")).strip():
        raise ValueError("Explain the adopt-or-keep decision.")
print(review["decision"], review["reason"])


## Topic — Explaining the Module Project: RAG Evidence Engine

**Theory and project explanation · 8 minutes**

Follow one inspectable evidence record from corpus provenance and frozen questions to retrieval improvement, grounded answers, refusal behavior, judge records and a justified extra experiment. The video uses illustrative counts and answers; it does not report a learner submission or a measured model run. Keep the warm-up notebook distinct from the capstone requirements below.


### Published competency evidence map

The supplied v2.0 rubric governs these criteria. The tutorial illustrates methods; the learner must supply their own actual project evidence. Review the must-pass criterion first.

| Criterion | Evidence | Met standard |
|---|---|---|
| C2.1 — Your corpus and gold questions are valid | D2.1 CORPUS; D2.2 GOLD; source provenance and frozen snapshot. | Include at least 100 real reviews, all four official documents verbatim with their IDs, and at least two authored documents containing testable facts and a cross-document link. Before indexing, freeze at least 20 corpus-based gold questions, including three multi-hop cases, with required source IDs and reference answers. Preserve the version record. |
| C2.2 — Your retriever improves under a fair comparison | D2.3 retrieve(); D2.6 evaluation; D2.7 upgrade experiment. | Beat retrieve_v0 on recall@4 using the same corpus, frozen gold questions and metric implementation. Save baseline/final recall at k = 1, 2, 4 and 8. Measure one upgrade beyond the query prefix on unchanged questions, show before/after results, and justify adopting or rejecting it. A rejected upgrade is valid evidence. |
| C2.3 **Must-pass** — Your answers are grounded and refuse unsupported claims | D2.4 rag_answer(); answer/hit pairs; five-question adversarial set and saved responses. | Return the answer and the actual retrieved hits. Citations resolve to those source IDs and support the associated claims. Preselect five genuinely unanswerable questions; at least four receive a refusal containing a REFUSAL_PHRASES phrase without an invented answer. Report all five outcomes; separately report any additional adversarial cases. |
| C2.4 — Your generation evaluation measures both axes | D2.5 scoring outputs; D2.6 evaluation and ledger; judge protocol. | Score faithfulness and answer relevance separately over the frozen gold set. State the judge or evaluation tool, scoring instructions, sample size and aggregation, and preserve representative answer-level results. Explain judge bias, especially shared generator/judge weights. Save the Stage 2 outputs and ledger; reference-substring matches alone do not establish faithfulness. |
| C2.5 — Your retrieval design choice is justified | Design note beside D2.7; one question and its linked source IDs. | Trace a question requiring linked facts across documents. Explain whether graph-assisted retrieval would help and identify its extraction, indexing and maintenance costs. Distinguish the notebook graph-hop illustration from full GraphRAG and KAG systems. Justify the retrieval approach used in RiskLens; implementing a complete graph framework is not required. |

All criteria must be Met for this competency to be Mastered. See [practice-to-capstone differences](../../docs/ASSESSMENT.md) and the [exact consolidated rubric](https://github.com/SarasAI-Institute/Building-Custom-AI-Capstone/blob/codex/competency-colab-refresh/RUBRICS.md). Preserve full outputs and disclose limitations; no metric compensates for missing required evidence.
